In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt

# 设备配置（CPU/GPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)  # 固定随机种子
np.random.seed(42)

#### 几何参数

In [2]:
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径

#### 材料热物性参数

In [3]:
# PCM（石蜡）
rho_s = 880.0    # 固相密度 (kg/m³)
rho_l = 760.0    # 液相密度 (kg/m³)
cp_s = 2180.0    # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0    # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4   # 固相导热系数 (W/(m·K))
lambda_l = 0.15  # 液相导热系数 (W/(m·K))
mu_l = 0.001     # 液相粘度 (kg/(m·s))
L = 255000.0     # 相变潜热 (J/kg)，论文中L=255kJ/kg
Tpc = 316.15     # 相变温度 (K)
DeltaT = 6.0     # 相变温度区间 (K)

# 高导热材料（铜）
rho_Cu = 8960.0  # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0    # 定压比热容 (J/(kg·K))

# 外壳材料（铝）- 仅用于边界，此处拓扑优化设计域为PCM+铜，铝不参与设计
rho_Al = 2719.0  # 密度 (kg/m³)
lambda_Al = 202.4  # 导热系数 (W/(m·K))
cp_Al = 879.0    # 定压比热容 (J/(kg·K))

#### 物理模型参数

In [4]:
Am = 1e5         # 糊状区常数
epsilon = 0.001  # 避免分母为0（论文方程4）
alpha = 2.1e-4   # PCM体胀系数 (1/K)
g = 9.81         # 重力加速度 (m/s²)

#### 拓扑优化参数

In [5]:
phi_total = 0.3  # 高导热材料体积比约束（论文预设值）
case = 3         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标

#### 训练超参数 

In [6]:
N_mass = 10000   # 质量守恒方程采样点数量
N_mom = 10000    # 动量方程采样点数量
N_heat = 10000   # 传热方程采样点数量
N_IC = 8000      # 初始条件采样点数量
N_BC1 = 3000     # 内管壁边界采样点数量
N_BC2 = 3000     # 外壳边界采样点数量
N_obj = 10000    # 优化目标采样点数量

# 损失项权重
lambda1 = 1.0    # PDE损失权重
lambda2 = 10.0   # IC/BC损失权重
lambda3 = 100.0  # 拓扑约束损失权重
lambda4 = 0.01   # 优化目标损失权重（初始较小，逐渐增加）

In [7]:
class ResidualBlock(nn.Module):
    """残差块：缓解深层网络梯度消失，提升表达能力"""
    def __init__(self, dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.activation = nn.Tanh()
    
    def forward(self, x):
        residual = x
        out = self.activation(self.fc1(x))
        out = self.fc2(out)
        return self.activation(out + residual) #残差相加后激活

In [8]:
class TopoPINN(nn.Module):
    def __init__(self, hidden_layers=6, hidden_dim=256):
        super(TopoPINN, self).__init__()
        
        # 主网络：处理随时间变化的物理场 (ux, uy, p, T)
        self.main_net = nn.Sequential(
            nn.Linear(3, hidden_dim),  # 输入(x,y,τ)
            nn.Tanh(),
            *[ResidualBlock(hidden_dim) for _ in range(hidden_layers)],
            nn.Linear(hidden_dim, 4)  # 输出ux, uy, p, T
        )
        
        # 拓扑网络：仅处理空间变量，输出材料密度场
        self.topo_net = nn.Sequential(
            nn.Linear(2, hidden_dim),  # 仅输入(x,y)
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),  # 输出设计变量ρ
            nn.Sigmoid()
        )
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """前向传播：x = [x_norm, y_norm, τ_norm]"""
        # 提取空间坐标
        x_space = x[:, 0:2]
        
        # 主网络输出
        main_out = self.main_net(x)
        ux = main_out[:, 0:1] * 0.01  # 速度尺度：0.01 m/s
        uy = main_out[:, 1:2] * 0.01
        p = main_out[:, 2:3] * 1000.0  # 压力尺度：1000 Pa
        T = main_out[:, 3:4] * 100.0 + 300.0  # 温度：300-400 K
        
        # 拓扑网络输出（仅依赖于空间坐标）
        rho_design = self.topo_net(x_space)
        
        return ux, uy, p, T, rho_design
    
    def compute_liquid_fraction(self, T):
        """计算液相率φ(T) - 论文方法"""
        # 相变温度区间
        T_lower = Tpc - DeltaT/2
        T_upper = Tpc + DeltaT/2
        
        # 线性相变模型
        phi = (T - T_lower) / (T_upper - T_lower)
        return torch.clamp(phi, 0.0, 1.0)

In [9]:
def compute_material_properties(T, rho_design):
    """
    计算混合材料的热物性参数（PCM+铜）
    """
    # 1. 计算液相率φ(T)
    phi = torch.clamp((T - (Tpc - DeltaT/2)) / DeltaT, 0.0, 1.0)
    
    # 2. PCM密度
    rho_PCM = rho_s + (rho_l - rho_s) * phi
    
    # 3. 混合密度（SIMP插值）
    # 使用ρ^3插值，鼓励0/1分布
    rho_design_pow = rho_design**3
    rho_total = rho_design_pow * rho_Cu + (1 - rho_design_pow) * rho_PCM
    
    # 4. PCM导热系数
    lambda_PCM = lambda_s + (lambda_l - lambda_s) * phi
    
    # 5. 混合导热系数（SIMP插值）
    lambda_total = rho_design_pow * lambda_Cu + (1 - rho_design_pow) * lambda_PCM
    
    # 6. 糊状区源项S_t(T)
    # 注意：当φ→0时，S_t→∞；当φ→1时，S_t→0
    S_t = Am * ((1.0 - phi)**2) / (phi**2 + epsilon)
    
    # 7. PCM粘度
    mu_PCM = mu_l + S_t * 1.0
    
    # 8. 混合粘度：铜为固体，取大值抑制流动
    mu_Cu_solid = 1e10
    mu_total = rho_design_pow * mu_Cu_solid + (1 - rho_design_pow) * mu_PCM
    
    # 运动粘度
    nu = mu_total / rho_total
    
    # 9. PCM比热容
    # 高斯函数D(T)
    sigma = DeltaT / 4.0
    D_T = torch.exp(-((T - Tpc) ** 2) / (sigma ** 2)) / (torch.sqrt(torch.tensor(np.pi)) * sigma)
    
    cp_PCM = cp_s + phi * (cp_l - cp_s) + L * D_T
    
    # 10. 混合比热容
    cp_total = rho_design_pow * cp_Cu + (1 - rho_design_pow) * cp_PCM
    
    # 11. 热扩散率
    a = lambda_total / (rho_total * cp_total + 1e-10)
    
    return {
        'rho': rho_total,
        'lambda': lambda_total,
        'mu': mu_total,
        'nu': nu,
        'cp': cp_total,
        'a': a,
        'S_t': S_t,
        'phi': phi
    }

In [10]:
def sample_collocation_points(N):
    """采样设计域内配点（x,y,τ）"""
    # 时间采样
    tau = np.random.uniform(0.0, 1e5, size=(N, 1))
    tau_norm = tau / 1e5
    
    # 空间采样（环形域）
    r = np.sqrt(np.random.uniform(r0**2, r1**2, size=(N, 1)))  # 均匀面积采样
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    # 归一化
    x_norm = x / r1
    y_norm = y / r1
    
    # 面积权重（用于积分）
    weight = r / np.mean(r)
    
    points = np.hstack([x_norm, y_norm, tau_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))

def sample_initial_condition(N):
    """采样初始条件点（τ=0）"""
    tau_norm = np.zeros((N, 1))
    
    r = np.sqrt(np.random.uniform(r0**2, r1**2, size=(N, 1)))
    theta = np.random.uniform(0.0, 2*np.pi, size=(N, 1))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    x_norm = x / r1
    y_norm = y / r1
    weight = r / np.mean(r)
    
    points = np.hstack([x_norm, y_norm, tau_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weight, dtype=torch.float32).to(device))

def sample_boundary(N1, N2):
    """采样边界点"""
    # 内管壁（Dirichlet边界）
    theta1 = np.random.uniform(0.0, 2*np.pi, size=(N1, 1))
    x1 = r0 * np.cos(theta1)
    y1 = r0 * np.sin(theta1)
    x1_norm = x1 / r1
    y1_norm = y1 / r1
    tau1_norm = np.random.uniform(0.0, 1.0, size=(N1, 1))
    bc1_points = np.hstack([x1_norm, y1_norm, tau1_norm])
    
    # 外壳（Neumann边界）
    theta2 = np.random.uniform(0.0, 2*np.pi, size=(N2, 1))
    x2 = r1 * np.cos(theta2)
    y2 = r1 * np.sin(theta2)
    x2_norm = x2 / r1
    y2_norm = y2 / r1
    tau2_norm = np.random.uniform(0.0, 1.0, size=(N2, 1))
    bc2_points = np.hstack([x2_norm, y2_norm, tau2_norm])
    
    return (torch.tensor(bc1_points, dtype=torch.float32).to(device),
            torch.tensor(bc2_points, dtype=torch.float32).to(device))

In [11]:
def compute_volume_constraint(rho_design, weights):
    """计算体积约束损失"""
    # 体积分数平均值
    rho_avg = torch.sum(rho_design * weights) / torch.sum(weights)
    
    # 惩罚超出约束的部分
    vol_residual = torch.relu(rho_avg - phi_total)
    return vol_residual**2 * 1000.0, rho_avg

def compute_topology_loss(rho_design, weights):
    """计算拓扑相关损失"""
    # 1. 取值范围约束 [0,1]
    bounds_loss = torch.mean(torch.relu(-rho_design)**2 + torch.relu(rho_design-1.0)**2)
    
    # 2. SIMP惩罚（鼓励二值化）
    p = 3.0
    simp_loss = torch.mean(rho_design**p * (1.0 - rho_design)**p)
    
    return bounds_loss + simp_loss * 10.0

def compute_pde_loss(model, points, weights):
    """计算PDE损失"""
    points.requires_grad_(True)
    ux, uy, p, T, rho_design = model(points)
    
    # 计算热物性
    props = compute_material_properties(T, rho_design)
    rho = props['rho']
    nu = props['nu']
    a = props['a']
    S_t = props['S_t']
    phi = props['phi']
    
    # 1. 质量守恒方程：∇·u = 0
    grad_ux = torch.autograd.grad(ux, points, grad_outputs=torch.ones_like(ux), 
                                  create_graph=True, retain_graph=True)[0]
    grad_uy = torch.autograd.grad(uy, points, grad_outputs=torch.ones_like(uy),
                                  create_graph=True, retain_graph=True)[0]
    
    dudx = grad_ux[:, 0:1]
    dvdy = grad_uy[:, 1:2]
    mass_residual = dudx + dvdy
    
    # 2. 动量守恒方程（x方向）
    convect_x = ux * dudx + uy * grad_ux[:, 1:2]
    
    grad_p = torch.autograd.grad(p, points, grad_outputs=torch.ones_like(p),
                                 create_graph=True, retain_graph=True)[0]
    dpdx = grad_p[:, 0:1]
    pressure_x = -dpdx / rho
    
    # 粘性项
    d2udx2 = torch.autograd.grad(dudx, points, grad_outputs=torch.ones_like(dudx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2udy2 = torch.autograd.grad(grad_ux[:, 1:2], points, grad_outputs=torch.ones_like(grad_ux[:, 1:2]),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_x = nu * (d2udx2 + d2udy2)
    
    # 糊状区源项
    source_x = -S_t * ux
    
    mom_residual_x = convect_x - pressure_x - viscous_x - source_x
    
    # 3. 动量守恒方程（y方向）
    convect_y = ux * grad_uy[:, 0:1] + uy * dvdy
    dpdy = grad_p[:, 1:2]
    pressure_y = -dpdy / rho
    
    d2vdx2 = torch.autograd.grad(grad_uy[:, 0:1], points, grad_outputs=torch.ones_like(grad_uy[:, 0:1]),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2vdy2 = torch.autograd.grad(dvdy, points, grad_outputs=torch.ones_like(dvdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_y = nu * (d2vdx2 + d2vdy2)
    
    source_y = -S_t * uy
    
    # 浮力项
    F_B = rho_l * alpha * g * (T - Tpc) * (1.0 - rho_design) * phi
    
    mom_residual_y = convect_y - pressure_y - viscous_y - source_y - F_B
    
    # 4. 能量方程
    grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                 create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    dTdtau = grad_T[:, 2:3] / 1e5  # 归一化时间导数
    
    convect_T = ux * dTdx + uy * dTdy
    
    d2Tdx2 = torch.autograd.grad(dTdx, points, grad_outputs=torch.ones_like(dTdx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2Tdy2 = torch.autograd.grad(dTdy, points, grad_outputs=torch.ones_like(dTdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    diffusive_T = a * (d2Tdx2 + d2Tdy2)
    
    energy_residual = dTdtau + convect_T - diffusive_T
    
    # 加权损失
    L_mass = torch.mean(mass_residual**2 * weights)
    L_momentum = torch.mean((mom_residual_x**2 + mom_residual_y**2) * weights)
    L_energy = torch.mean(energy_residual**2 * weights)
    
    return L_mass + L_momentum + L_energy, L_mass, L_momentum, L_energy

In [12]:
def compute_boundary_loss(model, bc1_points, bc2_points, heat_storage=True):
    """计算边界条件损失"""
    # 内管壁（Dirichlet边界）
    _, _, _, T_bc1, _ = model(bc1_points)
    Tw = 360.0 if heat_storage else 290.0  # 储热360K，释热290K
    L_bc1 = torch.mean((T_bc1 - Tw)**2)
    
    # 外壳（Neumann边界，绝热）
    bc2_points.requires_grad_(True)
    _, _, _, T_bc2, _ = model(bc2_points)
    
    grad_T = torch.autograd.grad(T_bc2, bc2_points, grad_outputs=torch.ones_like(T_bc2),
                                 create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    
    # 计算法向梯度（径向方向）
    x_real = bc2_points[:, 0:1] * r1
    y_real = bc2_points[:, 1:2] * r1
    r = torch.sqrt(x_real**2 + y_real**2 + 1e-10)
    dTdn = (x_real/r) * dTdx + (y_real/r) * dTdy
    
    L_bc2 = torch.mean(dTdn**2)
    
    return L_bc1 + L_bc2

In [13]:
def compute_initial_loss(model, ic_points, ic_weights, heat_storage=True):
    """计算初始条件损失"""
    _, _, _, T_ic, _ = model(ic_points)
    T0 = 290.0 if heat_storage else 360.0
    T_loss = torch.mean((T_ic - T0)**2 * ic_weights)
    
    # 初始速度为零
    ux, uy, _, _, _ = model(ic_points)
    u_loss = torch.mean((ux**2 + uy**2) * ic_weights)
    
    return T_loss + u_loss

In [14]:
def compute_objective_loss(model, case):
    """计算优化目标损失"""
    # 采样优化区域
    points, weights = sample_collocation_points(N_obj)
    points.requires_grad_(True)
    
    _, _, _, T, rho_design = model(points)
    weights = weights.unsqueeze(1)
    
    # 1. 平均温度
    T_avg = torch.sum(T * weights) / torch.sum(weights)
    
    # 2. 温度均方差
    T_var = torch.sum(((T - T_avg)**2) * weights) / torch.sum(weights)
    
    # 3. 火积耗散（论文方程16）
    grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                 create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    
    # 计算导热系数
    props = compute_material_properties(T, rho_design)
    lambda_total = props['lambda']
    
    # φ_g = ∫λ|∇T|² dA
    phi_g = torch.sum(lambda_total * (dTdx**2 + dTdy**2) * weights) / torch.sum(weights)
    
    # 根据case选择目标
    if case == 1:
        return T_avg**2, T_avg.item(), T_var.item(), phi_g.item()
    elif case == 2:
        return T_var, T_avg.item(), T_var.item(), phi_g.item()
    else:  # case 3: 多目标
        return T_avg**2 + T_var + phi_g**2, T_avg.item(), T_var.item(), phi_g.item()

In [15]:
def train_model(model, epochs=10000):
    """训练模型"""
    print("="*50)
    print("开始训练拓扑优化PINN模型")
    print("="*50)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=500, factor=0.5)
    
    # 初始化采样点
    colloc_points, colloc_weights = sample_collocation_points(N_mass)
    ic_points, ic_weights = sample_initial_condition(N_IC)
    bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
    
    best_loss = float('inf')
    
    for epoch in range(epochs):
        # 动态调整损失权重（逐渐增加优化目标权重）
        if epoch < 2000:
            current_lambda4 = 0.0
        elif epoch < 5000:
            current_lambda4 = lambda4 * 0.1
        else:
            current_lambda4 = lambda4
        
        # 前向传播计算各损失
        L_pde, L_mass, L_mom, L_energy = compute_pde_loss(model, colloc_points, colloc_weights)
        L_ic = compute_initial_loss(model, ic_points, ic_weights, heat_storage=True)
        L_bc = compute_boundary_loss(model, bc1_points, bc2_points, heat_storage=True)
        
        # 计算拓扑约束损失
        _, _, _, _, rho_design = model(colloc_points)
        L_vol, rho_avg = compute_volume_constraint(rho_design, colloc_weights)
        L_topo = compute_topology_loss(rho_design, colloc_weights)
        L_constraint = L_vol + L_topo
        
        # 计算优化目标损失
        L_obj, T_avg, T_var, phi_g = compute_objective_loss(model, case)
        
        # 总损失
        L_total = (lambda1 * L_pde + lambda2 * (L_ic + L_bc) + 
                   lambda3 * L_constraint + current_lambda4 * L_obj)
        
        # 反向传播
        optimizer.zero_grad()
        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step(L_total)
        
        # 记录最佳模型
        if L_total.item() < best_loss:
            best_loss = L_total.item()
            torch.save(model.state_dict(), f'topo_pinn_case{case}_best.pth')
        
        # 打印训练信息
        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}] | '
                  f'总损失: {L_total.item():.2e} | '
                  f'PDE: {L_pde.item():.2e} | '
                  f'IC/BC: {(L_ic+L_bc).item():.2e} | '
                  f'约束: {L_constraint.item():.2e} | '
                  f'目标: {L_obj.item():.2e}')
            
            print(f'  质量: {L_mass.item():.2e} | 动量: {L_mom.item():.2e} | '
                  f'能量: {L_energy.item():.2e} | 体积分数: {rho_avg.item():.3f}')
            
            print(f'  平均温度: {T_avg:.1f}K | 温度方差: {T_var:.2f} | '
                  f'火积耗散: {phi_g:.2e}')
            print("-"*80)
        
        # 每500轮重新采样
        if (epoch + 1) % 500 == 0:
            colloc_points, colloc_weights = sample_collocation_points(N_mass)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc1_points, bc2_points = sample_boundary(N_BC1, N_BC2)
    
    # 保存最终模型
    torch.save(model.state_dict(), f'topo_pinn_case{case}_final.pth')
    print("\n训练完成！模型已保存")


In [16]:
def evaluate_model(model):
    """评估模型性能"""
    print("\n" + "="*50)
    print("模型性能评估")
    print("="*50)
    
    model.eval()
    
    # 1. 计算完全熔化时间
    complete_time = None
    print("计算完全熔化时间...")
    for tau in range(0, 100000, 100):
        tau_norm = tau / 1e5
        points, weights = sample_collocation_points(2000)
        points[:, 2:] = tau_norm
        
        with torch.no_grad():
            _, _, _, T, _ = model(points)
            phi = model.compute_liquid_fraction(T)
            phi_avg = torch.sum(phi * weights) / torch.sum(weights)
            
            if phi_avg.item() >= 0.98:
                complete_time = tau
                break
    
    # 2. 计算平均储热容量
    avg_heat_storage = 0.0
    if complete_time:
        print(f"完全熔化时间: {complete_time}s")
        print("计算平均储热容量...")
        
        heat_fluxes = []
        for tau in range(0, complete_time, 100):
            tau_norm = tau / 1e5
            points, weights = sample_collocation_points(1000)
            points[:, 2:] = tau_norm
            points.requires_grad_(True)
            
            _, _, _, T, rho_design = model(points)
            props = compute_material_properties(T, rho_design)
            lambda_total = props['lambda']
            
            grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                         retain_graph=True)[0]
            dTdr = (points[:, 0:1]*r1 * grad_T[:, 0:1] + 
                    points[:, 1:2]*r1 * grad_T[:, 1:2]) / r1
            
            q = -lambda_total * dTdr
            q_avg = torch.sum(q * weights) / torch.sum(weights)
            heat_fluxes.append(q_avg.item())
        
        avg_heat_storage = np.mean(heat_fluxes)
        print(f"平均储热容量: {avg_heat_storage:.2f} W")
    
    # 3. 计算场协同角
    print("计算场协同角...")
    synergy_angles = []
    for tau in [200, 400, 600, 800]:
        tau_norm = tau / 1e5
        points, _ = sample_collocation_points(1000)
        points[:, 2:] = tau_norm
        points.requires_grad_(True)
        
        ux, uy, _, T, _ = model(points)
        grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                     retain_graph=True)[0]
        
        u_dot_gradT = ux * grad_T[:, 0:1] + uy * grad_T[:, 1:2]
        norm_u = torch.sqrt(ux**2 + uy**2 + 1e-10)
        norm_gradT = torch.sqrt(grad_T[:, 0:1]**2 + grad_T[:, 1:2]**2 + 1e-10)
        
        cos_theta = u_dot_gradT / (norm_u * norm_gradT + 1e-10)
        cos_theta = torch.clamp(cos_theta, -1.0, 1.0)
        theta = torch.acos(cos_theta) * 180 / np.pi
        
        synergy_angles.append(torch.mean(theta).item())
    
    avg_synergy = np.mean(synergy_angles)
    print(f"平均场协同角: {avg_synergy:.1f}°")
    
    return {
        'complete_melting_time': complete_time,
        'avg_heat_storage': avg_heat_storage,
        'avg_synergy_angle': avg_synergy
    }


In [17]:
def visualize_results(model, save_path="./results"):
    """可视化结果"""
    import os
    os.makedirs(save_path, exist_ok=True)
    
    # 生成网格
    r = np.linspace(r0, r1, 100)
    theta = np.linspace(0, 2*np.pi, 100)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 选择时间点
    time_points = [100, 300, 500, 751, 1000]
    
    for tau in time_points:
        tau_norm = tau / 1e5
        
        # 准备输入
        x_input = np.hstack([
            X.reshape(-1, 1) / r1,
            Y.reshape(-1, 1) / r1,
            np.full((10000, 1), tau_norm)
        ])
        
        x_tensor = torch.tensor(x_input, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            ux, uy, p, T, rho_design = model(x_tensor)
            phi = model.compute_liquid_fraction(T)
        
        # 转换为numpy
        T_grid = T.cpu().numpy().reshape(100, 100)
        phi_grid = phi.cpu().numpy().reshape(100, 100)
        rho_grid = rho_design.cpu().numpy().reshape(100, 100)
        
        # 绘图
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # 温度分布
        im1 = axes[0].contourf(X, Y, T_grid, levels=50, cmap='jet')
        axes[0].set_title(f'Temperature at τ={tau}s')
        axes[0].set_aspect('equal')
        plt.colorbar(im1, ax=axes[0])
        
        # 液相率
        im2 = axes[1].contourf(X, Y, phi_grid, levels=50, cmap='viridis')
        axes[1].set_title(f'Liquid Fraction at τ={tau}s')
        axes[1].set_aspect('equal')
        plt.colorbar(im2, ax=axes[1])
        
        # 拓扑结构
        im3 = axes[2].contourf(X, Y, rho_grid, levels=50, cmap='binary')
        axes[2].set_title(f'Topology at τ={tau}s (ρ≥0.5=Cu)')
        axes[2].set_aspect('equal')
        plt.colorbar(im3, ax=axes[2])
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f'result_{tau}s.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"可视化结果已保存到 {save_path}")

In [18]:
if __name__ == "__main__":
    # 初始化模型
    model = TopoPINN(hidden_layers=6, hidden_dim=256).to(device)
    
    print(f"使用Case {case}作为优化目标")
    print(f"设备: {device}")
    
    # 训练模型
    train_model(model, epochs=10000)
    
    # 加载最佳模型
    model.load_state_dict(torch.load(f'topo_pinn_case{case}_best.pth'))
    model.eval()
    
    # 评估模型
    metrics = evaluate_model(model)
    print("\n评估结果:")
    for key, value in metrics.items():
        if value is not None:
            print(f"{key}: {value}")
    
    # 可视化
    visualize_results(model, save_path=f"./results_case{case}")
    
    print("\n程序执行完成！")

使用Case 3作为优化目标
设备: cuda
开始训练拓扑优化PINN模型
Epoch [100/10000] | 总损失: 1.40e+09 | PDE: 1.40e+09 | IC/BC: 4.92e+05 | 约束: 6.39e-02 | 目标: 2.89e+16
  质量: 7.36e-11 | 动量: 1.40e+09 | 能量: 5.71e-09 | 体积分数: 0.242
  平均温度: -1699189.1K | 温度方差: 28866662027493376.00 | 火积耗散: 1.74e+04
--------------------------------------------------------------------------------
Epoch [200/10000] | 总损失: 1.00e+10 | PDE: 1.00e+10 | IC/BC: 4.74e+05 | 约束: 6.39e-02 | 目标: 2.59e+16
  质量: 2.12e-11 | 动量: 1.00e+10 | 能量: 2.60e-08 | 体积分数: 0.242
  平均温度: -1608392.5K | 温度方差: 25864091193049088.00 | 火积耗散: 1.75e+04
--------------------------------------------------------------------------------


KeyboardInterrupt: 